# Di-8-ANEPPS controls of Semenov et al. (2025), on DANDI:001272

Paper: Semenov, Kim, Silkuniene & Pakhomov (2025), "Excitation and polarization of isolated neurons
by high-frequency sine waves for temporal interference stimulation", Cell Reports Physical Science
6, 102660, doi:10.1016/j.xcrp.2025.102660.

Dataset: DANDI:001272, "The Use of Di8-ANEPPS to Evaluate Neuronal Sensitivity to Unmodulated Sine
Waves", version 0.250805.1358. The paper cites the dandiset without a version. Two versions are
published, 0.250701.1622 and 0.250805.1358; both hold the same 65 files with the same blobs (checked
below), so the most recent published version is pinned.

Claims: `claims/001272/10.1016-j.xcrp.2025.102660.yaml`, written from the NIH author manuscript
(PMC12320893) with its STAR Methods and supplement. Its header items F1 to F6 and O1 to O6 are cited
below by label; the helper `sinewave.py` says where every parameter comes from.

The FluoVolt reference group of the two `with_inputs` claims comes from the paper's other deposits,
DANDI:001032 (0.240527.1432), 001033 (0.240527.1432) and 001035 (0.240527.1432): their 250-ms
unmodulated 2-kHz sessions in NaCl solution. DANDI:001034 (0.240527.1431) is loaded only to show
that it duplicates 001032; DANDI:001271 (0.250805.1352) is used for one unit check.
The claims on the other deposits are tested in `fluovolt_elsewhere.ipynb`.

| Claim | Data | Here |
|---|---|---|
| figs1-eight-neurons | dataset | tested |
| figs1-aps-as-emission-decrease | dataset | tested |
| figs1-near-threshold-delayed-aps | dataset | tested |
| figs1-stronger-field-shorter-delay-sustained-firing | dataset | tested |
| figs1-culminates-in-sustained-depolarization | dataset | tested |
| figs1-noisier-than-fluovolt | with_inputs | tested |
| figs1-effects-same-as-fluovolt | with_inputs | tested |
| fig1, fig2, fig3, figs2 claims | elsewhere | `fluovolt_elsewhere.ipynb` |
| fig4, fig6, threshold-depolarization-estimate | elsewhere | blocked (see the attempt record) |

In [ ]:
import collections
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sinewave as sw

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
print("pinned:", sw.DANDISETS)

In [ ]:
# Both published versions of 001272 hold the same blobs.
assets = sw.list_assets("001272")
pinned = sw.DANDISETS["001272"]
sw.DANDISETS["001272"] = "0.250701.1622"
earlier = sw.list_assets("001272")
sw.DANDISETS["001272"] = pinned
print("files:", len(assets), "| earlier version files:", len(earlier),
      "| same blobs:", sorted(a["blob"] for a in assets) == sorted(a["blob"] for a in earlier))
D = sw.load_prepared(assets)
D.sort(key=lambda r: (r["subject"], r["start"]))
print("loaded", len(D), "sessions")

## Data check

What was loaded, and each check of the metadata against the values. The stimulus of each session
is known only from its `session_description` (frequency, amplitude in volts, and the files'
conversion "1V translates to 36 V/cm at the location of neurons"). The onset is not stored; F3 puts
it 50 ms into the session.

In [ ]:
meta = pd.DataFrame([dict(subject=r["subject"], start=pd.Timestamp(r["start"]), volts=r["volts"],
                          field=r["field"], freq_hz=r["freq_hz"], modulated=r["modulated"], ttx=r["ttx"],
                          rate=r["rate"], n_frames=r["n_frames"], mean_intensity=r["mean_intensity"],
                          frame0_ratio=r["frame0_ratio"], last_ratio=r["last_ratio"],
                          mask_a_px=r["mask_px"]["a"], eroded_empty=r["eroded_empty"],
                          md5=r["data_md5"], path=r["path"]) for r in D])
print("subjects:", meta.subject.nunique(), dict(meta.subject.value_counts().sort_index()))
print("frequency (Hz):", dict(meta.freq_hz.value_counts()), "| modulated:", int(meta.modulated.sum()),
      "| description says TTX:", int(meta.ttx.sum()))
print("frame rate (frames/s):", sorted(meta.rate.unique()), "| frames per file:", sorted(meta.n_frames.unique()),
      "| frame size: 16 x 16 pixels (the Methods give 64 x 64 after binning)")
print("recording length (ms):", round(1198 / meta.rate.iloc[0] * 1000, 1), "used, of",
      round(1200 / meta.rate.iloc[0] * 1000, 1))
for s, g in meta.groupby("subject"):
    print(f"{s}: {len(g)} sessions, {g.start.min():%Y-%m-%d %H:%M}-{g.start.max():%H:%M}, "
          f"amplitude {g.volts.min()}-{g.volts.max()} V = {g.field.min():.0f}-{g.field.max():.0f} V/cm, "
          f"{g.field.nunique()} fields, mean intensity {g.mean_intensity.min():.0f}-{g.mean_intensity.max():.0f} counts")
print("identical frame stacks (md5) within the dataset:", int(meta.md5.duplicated().sum()))
gaps = meta.sort_values("start").groupby("subject").start.diff().dt.total_seconds().dropna()
print(f"interval between sessions (s): median {gaps.median():.0f}, range {gaps.min():.0f}-{gaps.max():.0f}; "
      f"{(gaps < 20).sum()} of {len(gaps)} under the Methods' 20 s")
print(f"frame 0 / median frame: {meta.frame0_ratio.min():.2f}-{meta.frame0_ratio.max():.2f}; "
      f"last frame / median: {meta.last_ratio.min():.2f}-{meta.last_ratio.max():.2f} (both left out)")
print(f"O1 (a) Otsu ROI: {meta.mask_a_px.min()}-{meta.mask_a_px.max()} of 256 pixels; "
      f"eroding it by 2 pixels leaves nothing in {int(meta.eroded_empty.sum())} of {len(meta)} files, "
      "so reading (b) equals (a) there")
odd = meta[meta.mean_intensity < 0.5 * meta.groupby("subject").mean_intensity.transform("median")]
print("sessions with less than half their neuron's median intensity:", odd[["subject", "start", "volts", "mean_intensity"]].to_dict("records"))

In [ ]:
# The O1 (a) ROI against the frame: Otsu keeps one bright membrane edge of the soma, not the soma.
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
for i, s in enumerate(sorted(meta.subject.unique())):
    r = next(x for x in D if x["subject"] == s)
    rec = sw.read_session(next(a for a in assets if a["path"] == r["path"]))
    d, t = sw.frames(rec)
    img = d[: sw.onset_index(t)].mean(axis=0)
    axes[2 * i].imshow(img, cmap="gray"); axes[2 * i].set_title(f"{s}\nmean pre-stimulus frame", fontsize=8)
    axes[2 * i + 1].imshow(sw.roi_masks(img)["a"]); axes[2 * i + 1].set_title("O1 (a) ROI", fontsize=8)
    print(f"{s}: pre-stimulus frame {img.min():.0f}-{img.max():.0f} counts")
for a in axes:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

The ROI readings are therefore O1 (a), O1 (b) (identical to (a) wherever erosion empties it) and the
whole deposited 16 x 16 frame. The deposited frames are a crop around the soma (the Methods image
64 x 64 pixels), so the whole frame is the closest available reading of "the emission over the entire
cell soma"; it is run beside the claims file's two readings.

### Clock check

The files carry no stimulus timing, so the onset at 50 ms (F3) is checked against the responses: the
jump of the inverted dF over the 20 ms after a candidate onset, against the 20 ms before it, in every
session, at the true onset and at shifted ones. If the onset is right, the contrast peaks at 50 ms.

In [ ]:
def onset_contrast(recs, onset_ms, invert=True, roi="whole"):
    vals = []
    for r in recs:
        y = sw.dff(r["traces"][roi], r["t"], "a", invert=invert)
        t = r["t"]
        vals.append(y[(t >= onset_ms) & (t < onset_ms + 20)].mean() - y[(t >= onset_ms - 20) & (t < onset_ms)].mean())
    return np.array(vals)

shifts = [25, 30, 40, 50, 60, 70, 100, 150, 200, 250, 320]
cc = pd.DataFrame({f"{s} ms": [onset_contrast(D, s).mean()] for s in shifts}, index=["mean contrast (% dF)"])
print(cc.round(3).to_string())
c50 = onset_contrast(D, 50)
print(f"at 50 ms the contrast is positive in {np.sum(c50 > 0)} of {len(c50)} sessions")

## Session tables under every reading

For each combination of O1 (ROI), O2 (F0) and O3 (AP height), each session's APs are detected on the
inverted trace (F1) and, for figs1-aps-as-emission-decrease, on the trace as recorded. Reading names
are `roi/f0/height`.

In [ ]:
def rname(k):
    return "/".join(k)

TAB = {k: sw.session_table(D, *k, invert=True) for k in sw.READINGS}
RAW = {k: sw.session_table(D, *k, invert=False) for k in sw.READINGS}
TH = {k: sw.thresholds(TAB[k]) for k in sw.READINGS}
summary = pd.DataFrame([dict(reading=rname(k), sessions_with_aps=int((TAB[k].n_aps > 0).sum()),
                             aps=int(TAB[k].n_aps.sum()),
                             **{f"thr {s[:1]} (V/cm)": v for s, v in zip(TH[k].subject, TH[k].threshold)},
                             **{f"gap {s[:1]}": v for s, v in zip(TH[k].subject, TH[k].threshold - TH[k].next_lower)})
                        for k in sw.READINGS])
print(summary.to_string(index=False))

In [ ]:
# The traces under the whole-frame reading with F0 (a), APs of height reading (b) marked.
k = ("whole", "a", "b")
for s in sorted(meta.subject.unique()):
    rs = sorted([r for r in D if r["subject"] == s], key=lambda r: (r["field"], r["start"]))
    fig, axes = plt.subplots(len(rs), 1, figsize=(12, 0.9 * len(rs)), sharex=True)
    lo, hi = np.inf, -np.inf
    for ax, r in zip(axes, rs):
        y = sw.dff(r["traces"]["whole"], r["t"], "a", invert=True)
        ap = sw.detect_aps(y, r["t"], "b")
        ax.plot(r["t"], y, lw=0.5, c="k")
        ax.plot(ap["times"] + sw.ONSET_MS, ap["heights"], "v", c="tab:red", ms=4)
        ax.axvspan(sw.ONSET_MS, sw.ONSET_MS + sw.STIM_MS, color="tab:blue", alpha=0.08)
        ax.set_ylim(-4, 7)
        ax.text(1, 5, f"{r['field']:.0f} V/cm  {r['start'][11:19]}", fontsize=7)
        ax.set_yticks([0, 5])
        lo, hi = min(lo, y.min()), max(hi, y.max())
    axes[-1].set_xlabel("time (ms)"); axes[0].set_title(f"{s}: inverted dF (%), whole frame")
    plt.tight_layout(); plt.show()
    print(f"{s}: plotted dF range {lo:.1f} to {hi:.1f} %; axes -4 to 7 %")

## figs1-eight-neurons

> The Di-8-ANEPPS control experiments were made on eight neurons, at field strengths tested in a
> random sequence.

Criterion: neurons counted from the dataset's subjects, sessions or cell identifiers, each recording
once; expected 8, tolerance 0. The order of fields is reported, not judged.

In [ ]:
n_neurons = meta.subject.nunique()
print("neurons (subjects) in the dataset:", n_neurons, "| expected 8")
print("identical recordings:", int(meta.md5.duplicated().sum()))
for s, g in meta.sort_values("start").groupby("subject"):
    f = g.field.values
    rises = np.sum(np.diff(f) > 0); falls = np.sum(np.diff(f) < 0)
    print(f"{s}: field order (V/cm) {np.round(f).astype(int).tolist()}; rises {rises}, falls {falls}, "
          f"monotonic non-decreasing: {bool(np.all(np.diff(f) >= 0))}")

**Result: not reproduced.** The dataset holds 3 neurons (subjects 2, 3 and 4 of 2024-08-06), not 8,
with no recording deposited twice. No subject's fields rise monotonically over its sessions, so the
sequences are mixed; subjects 2 and 4 mostly escalate and subject 3 jumps between low and high fields.

## figs1-aps-as-emission-decrease

> With Di-8-ANEPPS, APs appear as brief reductions of emission, opposite in sign to FluoVolt.

Criterion: per neuron, over its sessions at fields above its threshold (F4), AP detection (O3) on the
trace as recorded and on the inverted trace; reproduced when in every neuron with APs the inverted
trace yields more APs than the trace as recorded in more than 75% of its sessions with APs.

In [ ]:
rows = []
for k in sw.READINGS:
    inv, raw, th = TAB[k], RAW[k], dict(zip(TH[k].subject, TH[k].threshold))
    ok_all, n_neur = True, 0
    for s in sorted(inv.subject.unique()):
        sel = (inv.subject == s) & (inv.field > th[s])
        ni, nr = inv.n_aps[sel].values, raw.n_aps[sel].values
        has = (ni > 0) | (nr > 0)
        if has.sum() == 0:
            continue
        n_neur += 1
        frac = float(np.mean(ni[has] > nr[has]))
        ok_all &= frac > 0.75
        rows.append(dict(reading=rname(k), subject=s, sessions_above_thr=int(sel.sum()), sessions_with_aps=int(has.sum()),
                         aps_inverted=int(ni.sum()), aps_as_recorded=int(nr.sum()), frac_more_inverted=round(frac, 2)))
    rows.append(dict(reading=rname(k), subject="ALL", sessions_with_aps=n_neur, frac_more_inverted=None, ok=ok_all))
c2 = pd.DataFrame(rows)
print(c2.to_string(index=False))
ok2 = c2[c2.subject == "ALL"].ok
print(f"criterion met under {int(ok2.sum())} of {len(ok2)} readings")
# The sign of the largest brief deflection, reported.
sign = []
for r in D:
    y = sw.dff(r["traces"]["whole"], r["t"], "a")
    w = (r["t"] >= 50) & (r["t"] < 300)
    yy = y[w] - np.convolve(y, np.ones(31) / 31, mode="same")[w]   # brief deflections only, for this report
    sign.append(np.sign(yy[np.argmax(np.abs(yy))]))
print("largest brief deflection during the stimulus (as recorded, minus a 10-ms running mean): negative in",
      int(np.sum(np.array(sign) < 0)), "of", len(sign), "sessions")

**Result: reproduced.** Under all 12 readings, every neuron with APs had more APs on the inverted
trace than on the trace as recorded in all of its sessions with APs (fraction 1.00 in every neuron),
and no AP was detected on the trace as recorded. The largest brief deflection during the stimulus,
as recorded, is negative in 57 of 65 sessions.

## figs1-near-threshold-delayed-aps

> Near threshold, 2-kHz sine waves elicited one or a few delayed APs in Di-8-ANEPPS neurons.

Criterion: per neuron, its near-threshold sessions (O4 (a) at the threshold field, (b) fields up to
10% above it); the median session has 1 to 3 APs and its first AP is delayed (O5 (a) at least 100 ms
after onset, (b) later than the median first-AP latency of the neuron's sessions at higher fields).
Reproduced when this holds in more than half of the neurons with APs.

In [ ]:
def near_threshold(tab, th, o4):
    out = []
    for s in sorted(tab.subject.unique()):
        g = tab[tab.subject == s]
        t0 = th[s]
        if np.isnan(t0):
            continue
        near = g[g.field == t0] if o4 == "a" else g[(g.field >= t0) & (g.field <= 1.1 * t0)]
        higher = g[(g.field > near.field.max()) & (g.n_aps > 0)]
        out.append(dict(subject=s, threshold=t0, near_sessions=len(near), median_aps=float(np.median(near.n_aps)),
                        aps=near.n_aps.tolist(), median_first_ms=float(np.nanmedian(near.first_ms)),
                        higher_median_first_ms=float(np.nanmedian(higher.first_ms)) if len(higher) else np.nan))
    return pd.DataFrame(out)

rows, detail = [], {}
for k in sw.READINGS:
    th = dict(zip(TH[k].subject, TH[k].threshold))
    for o4 in "ab":
        nt = near_threshold(TAB[k], th, o4)
        detail[(k, o4)] = nt
        few = (nt.median_aps >= 1) & (nt.median_aps <= 3)
        for o5 in "ab":
            delayed = nt.median_first_ms >= 100 if o5 == "a" else nt.median_first_ms > nt.higher_median_first_ms
            n_ok = int((few & delayed).sum())
            rows.append(dict(reading=rname(k), O4=o4, O5=o5, neurons_with_aps=len(nt), neurons_meeting=n_ok,
                             reproduced=n_ok > len(nt) / 2))
c3 = pd.DataFrame(rows)
print(c3.pivot_table(index="reading", columns=["O4", "O5"], values="neurons_meeting").to_string())
print(f"reproduced under {int(c3.reproduced.sum())} of {len(c3)} reading combinations")
print("by O5:", c3.groupby("O5").reproduced.agg(["sum", "count"]).to_dict("index"))
print("by height reading:", c3.assign(h=c3.reading.str[-1]).groupby("h").reproduced.agg(["sum", "count"]).to_dict("index"))
for key in [(("whole", "a", "b"), "a"), (("a", "a", "b"), "a"), (("whole", "a", "a"), "a")]:
    print(f"\nper neuron, reading {rname(key[0])}, O4 ({key[1]}):")
    print(detail[key].round(1).to_string(index=False))

**Result: partially reproduced, resting on an assumption.** The criterion is met in 12 of 48 reading
combinations, all under O5 (b) and none under O5 (a), the 100-ms delay the paper gives near threshold
for FluoVolt, which this record adopts. Near threshold, subject 3 fires single APs (median 1) whose
median latency is 63 ms after the onset, against 5 to 8 ms at higher fields, delayed relative to its
own stronger responses but under 100 ms; subject 4 fires 1 or 2 APs at a median 167 to 212 ms;
subject 2's median threshold session has no AP under these readings. So one or a few APs near
threshold hold in two of three neurons, and a delay of at least 100 ms in one.

## figs1-stronger-field-shorter-delay-sustained-firing

> In Di-8-ANEPPS neurons, increasing the field strength reduced the delay to the first AP and
> induced sustained firing.

Criterion: per neuron with sessions with APs at 3 or more fields, the Spearman correlation of first-AP
latency with field; reproduced when negative in more than half of those neurons and more than half of
all neurons show sustained firing (O6) in some session. F5 rates are reported.

In [ ]:
rows = []
for k in sw.READINGS:
    tab = TAB[k]
    rhos, sf = [], []
    for s in sorted(tab.subject.unique()):
        g = tab[tab.subject == s]
        ga = g[g.n_aps > 0]
        rho = stats.spearmanr(ga.field, ga.first_ms) if ga.field.nunique() >= 3 else None
        if rho is not None:
            rhos.append(rho.statistic)
        sf.append(bool(g.sust_firing.any()))
        rows.append(dict(reading=rname(k), subject=s, fields_with_aps=int(ga.field.nunique()),
                         rho=None if rho is None else round(rho.statistic, 2),
                         p=None if rho is None else round(rho.pvalue, 3), sustained_firing=sf[-1],
                         max_rate_hz=round(np.nanmax(g.rate_hz), 1) if g.rate_hz.notna().any() else None))
    neg = np.sum(np.array(rhos) < 0)
    rows.append(dict(reading=rname(k), subject="ALL", fields_with_aps=len(rhos), rho=int(neg), sustained_firing=int(sum(sf)),
                     max_rate_hz=None, reproduced=(len(rhos) > 0 and neg > len(rhos) / 2) and sum(sf) > len(sf) / 2))
c4 = pd.DataFrame(rows)
print(c4[c4.subject != "ALL"].to_string(index=False))
allc4 = c4[c4.subject == "ALL"][["reading", "fields_with_aps", "rho", "sustained_firing", "reproduced"]]
allc4.columns = ["reading", "neurons_3plus_fields", "negative_rho", "neurons_sustained_firing", "reproduced"]
print(allc4.to_string(index=False))
print(f"reproduced under {int(allc4.reproduced.sum())} of {len(allc4)} readings; "
      f"under height (b): {int(allc4[allc4.reading.str.endswith('b')].reproduced.sum())} of 6")
k = ("whole", "a", "b")
print("\nF5 maximum firing rate against field, reading whole/a/b:")
print(TAB[k].groupby(["subject", "field"]).rate_hz.median().round(1).unstack(0).to_string())

**Result: reproduced, resting on an assumption.** The criterion is met under 10 of 12 readings and
under all 6 with the height reading (b), which this record adopts for Di-8-ANEPPS: the latency
correlates negatively with field in more than half of the neurons with APs at 3 or more fields (2 of
2, 3 of 3, or 2 of 3), and 2 or 3 of the 3 neurons show sustained firing. Most single correlations are
not significant (p from 0.004 to 0.91). It fails under the 5% height (a) with the whole-frame ROI,
where few Di-8-ANEPPS APs reach 5%.

## figs1-culminates-in-sustained-depolarization

> In Di-8-ANEPPS neurons, the strongest fields replaced firing with sustained depolarization
> (excitation followed by inhibition of firing).

Criterion: per neuron, its sessions at its highest field; reproduced when more than half of the
neurons show sustained depolarization (O6) at their highest field and sustained firing or more APs at
some lower field. O6 levels (a) 5 SD of the trace before onset, (b) half the neuron's median AP
height. A neuron counts when most of its highest-field sessions show it (a standard choice).

In [ ]:
rows = []
for k in sw.READINGS:
    tab = TAB[k]
    for lev in "ab":
        n_ok, per = 0, []
        for s in sorted(tab.subject.unique()):
            g = tab[tab.subject == s]
            top, low = g[g.field == g.field.max()], g[g.field < g.field.max()]
            sd = top[f"sust_depol_{lev}"].mean() > 0.5
            more = bool(low.sust_firing.any() or (low.n_aps.max() > top.n_aps.max()))
            n_ok += sd and more
            per.append(f"{s[:1]}:{'D' if sd else '-'}{'F' if more else '-'}")
        rows.append(dict(reading=rname(k), level=lev, neurons=" ".join(per), meeting=n_ok, reproduced=n_ok > 1.5))
c5 = pd.DataFrame(rows)
print(c5.to_string(index=False))
print(f"reproduced under {int(c5.reproduced.sum())} of {len(c5)} combinations")
# Variant: AP height as prominence, so that a depolarized baseline does not make noise count as APs.
rows = []
for k in sw.READINGS:
    tab = sw.session_table(D, *k, invert=True, height_ref="prominence")
    for lev in "ab":
        n_ok = 0
        for s in sorted(tab.subject.unique()):
            g = tab[tab.subject == s]
            top, low = g[g.field == g.field.max()], g[g.field < g.field.max()]
            n_ok += (top[f"sust_depol_{lev}"].mean() > 0.5) and bool(low.sust_firing.any() or (low.n_aps.max() > top.n_aps.max()))
        rows.append(dict(reading=rname(k), level=lev, meeting=n_ok))
c5p = pd.DataFrame(rows)
print("variant, height as prominence: neurons meeting per reading and level:",
      c5p.pivot(index="reading", columns="level", values="meeting").to_dict("index"))
k = ("whole", "a", "b")
t = TAB[k]
print("\nlast 100 ms of the stimulus, reading whole/a/b, per neuron and field: mean inverted dF (%) | APs there")
print(t.groupby(["subject", "field"]).agg(late_mean=("late_mean", "mean"), late_aps=("late_aps", "mean"),
                                          pre_sd=("pre_sd", "mean")).round(2).to_string())

**Result: not reproduced.** At most 1 of the 3 neurons meets the criterion under any of the 24
combinations, where more than half are needed. Subjects 2 and 4 were not stimulated above 234 V/cm,
close to their thresholds, and never reach sustained depolarization. Subject 3, stimulated up to
504 V/cm, meets it only with the Otsu ROI and level (b), or under the 5% height; with the whole frame
and height (b) it still fires 3 APs in the last 100 ms at 504 V/cm, while its late inverted dF is
2.9 to 3.3% at 396 to 504 V/cm, against 1.4 to 2.4% at most lower fields (3.2% at 180 V/cm). The inverted trace drifts up in
every session from bleaching (1.4 to 1.7% late in subject 4 at fields that evoke no sustained
response), which the O6 levels do not remove. Taking the AP height as prominence, so that noise on a
depolarized baseline is not counted as APs, also gives at most 1 of 3 neurons.

## FluoVolt reference group (outside this dataset)

The `with_inputs` claims compare these neurons with FluoVolt neurons stimulated by 250-ms unmodulated
2-kHz sine waves in NaCl solution (claims file: DANDI:001030 and 001032 to 001035, assigned from
their titles). 001030 holds only 2-s ramped waves in sodium gluconate, so none of its sessions fit.
001034 is checked against 001032 first.

In [ ]:
def two_khz_unmodulated(ds):
    return [a for a in sw.list_assets(ds) if "_ses-2kHz-sine-stimulation" in a["path"]]

F = {ds: sw.load_prepared(two_khz_unmodulated(ds)) for ds in ("001032", "001033", "001035")}
dup = sw.load_prepared(two_khz_unmodulated("001034"))
h32 = {r["data_md5"] for r in F["001032"]}
print("001034 2-kHz sessions:", len(dup), "| identical frame stacks in 001032:", sum(r["data_md5"] in h32 for r in dup))
for ds, recs in F.items():
    m = pd.DataFrame([dict(subject=r["subject"], freq=r["freq_hz"], mod=r["modulated"], pp=r["peak_to_peak"],
                           naglu=r["nagluconate"], n=r["n_frames"], v=r["volts"]) for r in recs])
    print(f"{ds}: {len(recs)} sessions, {m.subject.nunique()} neurons, frequencies {sorted(m.freq.unique())}, "
          f"modulated {int(m['mod'].sum())}, amplitude stated as peak-to-peak {int(m.pp.sum())}, "
          f"frames per file {dict(m.n.value_counts())}, {m.v.min()}-{m.v.max()} V")
FV = [dict(r, dataset=ds, subject=f"{ds}:{r['subject']}") for ds, recs in F.items() for r in recs
      if r["freq_hz"] == 2000 and not r["modulated"]]
print("FluoVolt 2-kHz unmodulated sessions used:", len(FV), "from", len({r['subject'] for r in FV}), "neurons")

## figs1-noisier-than-fluovolt

> Di-8-ANEPPS traces were noisier than FluoVolt traces, as the dye has a lower voltage sensitivity.

Criterion: per neuron, the signal to noise ratio is the median AP height over the SD of dF before the
onset; readings of "noisier" (a) that ratio, (b) the SD itself. Reproduced when the Di-8-ANEPPS median
is worse than the FluoVolt median under both. A two-sided unpaired t test (F6) is reported, and the
median Di-8-ANEPPS AP height beside the FluoVolt 11.3% (Methods).

In [ ]:
FTAB = {k: sw.session_table(FV, *k, invert=False) for k in sw.READINGS}

def per_neuron_noise(tab):
    g = tab.explode("ap_heights")
    med_h = g.groupby("subject").ap_heights.apply(lambda s: float(np.nanmedian(s.astype(float))) if s.notna().any() else np.nan)
    sd = tab.groupby("subject").pre_sd.median()
    return pd.DataFrame({"median_ap_height": med_h, "pre_sd": sd, "snr": med_h / sd})

rows = []
for k in sw.READINGS:
    di, fv = per_neuron_noise(TAB[k]), per_neuron_noise(FTAB[k])
    snr_t = stats.ttest_ind(di.snr.dropna(), fv.snr.dropna())
    sd_t = stats.ttest_ind(di.pre_sd, fv.pre_sd)
    rows.append(dict(reading=rname(k), di8_snr=di.snr.median(), fv_snr=fv.snr.median(), p_snr=snr_t.pvalue,
                     di8_sd=di.pre_sd.median(), fv_sd=fv.pre_sd.median(), p_sd=sd_t.pvalue,
                     di8_height=di.median_ap_height.median(), fv_height=fv.median_ap_height.median(),
                     n_di8=int(di.snr.notna().sum()), n_fv=int(fv.snr.notna().sum()),
                     worse_both=(di.snr.median() < fv.snr.median()) and (di.pre_sd.median() > fv.pre_sd.median())))
c6 = pd.DataFrame(rows)
print(c6.round(3).to_string(index=False))
print(f"Di-8-ANEPPS worse under both measures in {int(c6.worse_both.sum())} of {len(c6)} readings")
k = ("whole", "a", "b")
print("\nper neuron, reading whole/a/b, Di-8-ANEPPS:")
print(per_neuron_noise(TAB[k]).round(2).to_string())
print("FluoVolt (summary over neurons):")
print(per_neuron_noise(FTAB[k]).describe().round(2).to_string())

**Result: partially reproduced.** The criterion requires both measures and is met under none of the
12 readings. By the signal to noise ratio (a) the Di-8-ANEPPS median is lower than FluoVolt's under 10
of 12 readings (whole frame: 7.4 to 9.7 against 12.6 to 12.9, t test p from 0.002 to 0.13), because its
median AP height is about half (4.1 to 6.6% against 10.4 to 13.5%; the FluoVolt value is near the
paper's typical 11.3%). By the SD before the onset (b) the Di-8-ANEPPS traces are less noisy, not more
(0.55 against 0.88% with the whole frame, p < 0.001; 1.24 against 1.57 or 3.05% with the Otsu ROI).

## figs1-effects-same-as-fluovolt

> Overall, the effects of 2-kHz sine waves in Di-8-ANEPPS neurons did not differ from those with
> FluoVolt, so the excitation by unmodulated sine waves is not a sensitization by FluoVolt.

Criterion: the response pattern holds when figs1-near-threshold-delayed-aps,
figs1-stronger-field-shorter-delay-sustained-firing and figs1-culminates-in-sustained-depolarization
are reproduced; the 1-AP thresholds (F4) of the Di-8-ANEPPS neurons are compared with those of the
FluoVolt neurons by a two-sided unpaired t test at p < 0.05. Reproduced when the pattern holds and the
test finds no difference.

The files state amplitudes differently: 001032 to 001035 give "peak-to-peak" volts, 001272 "amplitude"
in volts, 001030 "one polarity; peak-to-peak would be two-fold larger", all with 36 V/cm per volt. Two
readings are run: (i) the volts as stated, times 36, in both groups; (ii) the FluoVolt volts halved
(if 001272 states one polarity). DANDI:001271, FluoVolt recorded in 2024 with the same description
wording as 001272, gives a check of which scale 001272's "amplitude" is on.

In [ ]:
rows, diffs = [], []
for k in sw.READINGS:
    di = TH[k].dropna(subset=["threshold"])
    fv = sw.thresholds(FTAB[k]).dropna(subset=["threshold"])
    for unit, fvf in (("i", fv.threshold.values), ("ii", fv.threshold.values / 2)):
        tt = stats.ttest_ind(di.threshold.values, fvf)
        d = di.threshold.mean() - fvf.mean()
        sp = np.sqrt(((len(di) - 1) * di.threshold.var() + (len(fvf) - 1) * fvf.var()) / (len(di) + len(fvf) - 2))
        se = sp * np.sqrt(1 / len(di) + 1 / len(fvf))
        tc = stats.t.ppf(0.975, len(di) + len(fvf) - 2)
        rows.append(dict(reading=rname(k), unit=unit, n_di8=len(di), di8_mean=di.threshold.mean(),
                         n_fv=len(fvf), fv_mean=fvf.mean(), diff=d, ci_lo=d - tc * se, ci_hi=d + tc * se,
                         p=tt.pvalue, one_sample_vs_57_7_p=stats.ttest_1samp(di.threshold, 57.7).pvalue,
                         differ=tt.pvalue < 0.05))
c7 = pd.DataFrame(rows)
print(c7.round(3).to_string(index=False))
for u in ("i", "ii"):
    s = c7[c7.unit == u]
    print(f"unit reading ({u}): difference at p < 0.05 in {int(s.differ.sum())} of {len(s)} readings")
print("FluoVolt fv_mean under (i) is to be compared with the paper's 57.7 +/- 3.6 V/cm (not judged).")

In [ ]:
# Scale check: 2-kHz thresholds in volts of the FluoVolt vehicle sessions of 001271 (same 2024 wording
# as 001272), against 001032 to 001035 (stated peak-to-peak). Same reading as the main table.
ttx_assets = sw.list_assets("001271")
T71 = sw.load_prepared(ttx_assets)
k = ("whole", "a", "b")
t71 = sw.session_table([r for r in T71 if not r["ttx"] and not r["sham"]], *k, invert=False)
th71 = sw.thresholds(t71)
thfv = sw.thresholds(FTAB[k])
print(f"001271 vehicle, 'amplitude' volts: n = {th71.threshold_volts.notna().sum()}, "
      f"mean 1-AP threshold {th71.threshold_volts.mean():.2f} V (median {th71.threshold_volts.median():.2f})")
print(f"001032/33/35, 'peak-to-peak' volts: n = {thfv.threshold_volts.notna().sum()}, "
      f"mean {thfv.threshold_volts.mean():.2f} V (median {thfv.threshold_volts.median():.2f})")
print("001272 Di-8-ANEPPS, 'amplitude' volts:", TH[k].threshold_volts.tolist())
print("t test 001271 against 001032/33/35 volts: p =", round(stats.ttest_ind(th71.threshold_volts.dropna(), thfv.threshold_volts.dropna()).pvalue, 3))
print("pattern claims (from the sections above): near-threshold", "partially reproduced;",
      "stronger field", "reproduced under height (b);", "sustained depolarization", "not reproduced")

**Result: partially reproduced, resting on an assumption.** Under unit reading (i), the volts as the
files state them, which this record adopts, the t test finds no difference between the Di-8-ANEPPS
(n = 3, mean 134 to 152 V/cm) and FluoVolt (n = 18 to 27, mean 145 to 162 V/cm) 1-AP thresholds under
all 12 readings (p from 0.72 to 0.98; difference -14 to 7 V/cm, 95% intervals about -90 to 80 V/cm).
The 001271 check supports (i): its FluoVolt thresholds in the 2024 "amplitude" volts (mean 3.70 V)
do not differ from those of 001032 to 001035 in "peak-to-peak" volts (4.20 V, p = 0.29). Under reading
(ii) the Di-8-ANEPPS thresholds are higher in all 12 readings (p from 0.001 to 0.010). The response
pattern holds only in part: the stronger-field claim is reproduced, the near-threshold claim only in
part, and the sustained-depolarization claim is not. The FluoVolt mean under (i), about 150 V/cm, is
far above the paper's 57.7 V/cm.

## Summary

| Claim | Outcome |
|---|---|
| figs1-eight-neurons | not reproduced (3 neurons) |
| figs1-aps-as-emission-decrease | reproduced |
| figs1-near-threshold-delayed-aps | partially reproduced (assumption) |
| figs1-stronger-field-shorter-delay-sustained-firing | reproduced (assumption) |
| figs1-culminates-in-sustained-depolarization | not reproduced |
| figs1-noisier-than-fluovolt (with_inputs) | partially reproduced |
| figs1-effects-same-as-fluovolt (with_inputs) | partially reproduced (assumption) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: the soma outline is not deposited, so three ROI readings
are run; AP detection, delay, sustained firing and sustained depolarization follow the claims file's
readings, since the paper describes Figure S1 in words only; the Di-8-ANEPPS AP height reading (b),
5 SD of the trace before onset, is adopted where readings disagree, as the paper says this dye is
less sensitive than FluoVolt and its 5% level was set for FluoVolt.

What was found about the dataset: it holds 3 neurons, where the paper reports 8; the frames are
16 x 16 pixels, where the Methods give 64 x 64; the stimulus onset is not stored; the amplitude is
given in volts without saying whether it is peak or peak-to-peak, and the files of the paper's
deposits apply the same 36 V/cm per volt to peak-to-peak and one-polarity values; every file's
description says the experiment was a vehicle control for tetrodotoxin, which this experiment was
not; the first frame of every file is an artefact; several sessions are under 20 s apart, where the
Methods give 20 to 60 s; DANDI:001034 duplicates DANDI:001032 file for file.